In [21]:
import sys
sys.path.append('../')

from ingestion import load_faq_data, build_index
from rag_helper import RAGBase
from openai import OpenAI
from dotenv import load_dotenv
load_dotenv()

True

In [3]:
documents = load_faq_data()
index = build_index(documents)
    
# Initialize RAG helper with the index and OpenAI client
openai_client = OpenAI()

assistant = RAGBase(
        index=index,
        llm_client=openai_client,
    )

Loaded 6 courses
Fetching: https://datatalks.club/faq//json/data-engineering-zoomcamp.json
Added 402 documents from Data Engineering Zoomcamp (course: data-engineering-zoomcamp)
Fetching: https://datatalks.club/faq//json/stock-markets-analytics-zoomcamp.json
Added 93 documents from Stock Markets Analytics Zoomcamp (course: stock-markets-analytics-zoomcamp)
Fetching: https://datatalks.club/faq//json/ai-dev-tools-zoomcamp.json
Added 41 documents from AI Dev Tools Zoomcamp (course: ai-dev-tools-zoomcamp)
Fetching: https://datatalks.club/faq//json/llm-zoomcamp.json
Added 79 documents from LLM Zoomcamp (course: llm-zoomcamp)
Fetching: https://datatalks.club/faq//json/mlops-zoomcamp.json
Added 255 documents from MLOps Zoomcamp (course: mlops-zoomcamp)
Fetching: https://datatalks.club/faq//json/machine-learning-zoomcamp.json
Added 472 documents from ML Zoomcamp (course: machine-learning-zoomcamp)
Total documents loaded: 1342
Building index with 1342 documents
Text fields: ['question', 'sectio

In [4]:
question = "How can I get a certificate of completion for the course?"
answer = assistant.rag(question)
answer

[SEARCH] Query: How can I get a certificate of completion for the course?, Course: llm-zoomcamp, Num Results: 5
[SEARCH] Found 5 results
[BUILD_CONTEXT] Building context from 5 search results
[BUILD_CONTEXT] Context built with length: 1664


'You can get a certificate only if you finish the course with a **live cohort** and **pass the Capstone project**. \n\nSelf-paced mode does **not** include certificates, because you need to **peer-review 3 capstones** after submitting your project, and that’s only possible while the course is running and submissions are open.'

In [5]:
question_1 = "How do I run Ollama locally?"
answer = assistant.rag(question_1)
answer

[SEARCH] Query: How do I run Ollama locally?, Course: llm-zoomcamp, Num Results: 5
[SEARCH] Found 5 results
[BUILD_CONTEXT] Building context from 5 search results
[BUILD_CONTEXT] Context built with length: 2674


'To run Ollama locally, first install it from [https://ollama.com/download](https://ollama.com/download) for your operating system.\n\nThen open a terminal and run:\n\n```bash\nollama run llama3\n```\n\nThis will download the LLaMA 3 model, start it locally, and give you a chat-like interface.\n\nTo check that the local server is running, you can also use:\n\n```bash\ncurl http://localhost:11434\n```\n\nIf you want to use it from Python, install the client with:\n\n```bash\npip install ollama\n```'

In [6]:
question_2 = "How do I run Olama locally?"
answer = assistant.rag(question_2)
answer

[SEARCH] Query: How do I run Olama locally?, Course: llm-zoomcamp, Num Results: 5
[SEARCH] Found 5 results
[BUILD_CONTEXT] Building context from 5 search results
[BUILD_CONTEXT] Context built with length: 3022


"I don't know."

# Search tool for agentic RAG

In [10]:
# earch function for agentic RAG
def search(query):
    boost_dict = {"question": 3.0, "section": 0.5}
    filter_dict = {"course": "llm-zoomcamp"}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )

In [11]:
# Search tool for agentic RAG
search_tool = {
    "type": "function",
    "name": "search",
    "description": "Search the FAQ database for entries matching the given query.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Search query text to look up in the course FAQ."
            }
        },
        "required": ["query"],
        "additionalProperties": False
    }
}

In [12]:
messages = [
    {
        "role": "user",
        "content": question_2
    }
]

In [13]:
response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)

response.output

[ResponseFunctionToolCall(arguments='{"query":"Olama locally run install model serve local FAQ"}', call_id='call_9Nl5VZeLEQBEZO7u1YyYLz3M', name='search', type='function_call', id='fc_0d49fe1f4657922a006a29cd447d0881a2ac6672a1bf43f323', namespace=None, status='completed')]

In [14]:
import json
# Extract the arguments from the tool call and execute the search function
call = response.output[0]
args = json.loads(call.arguments)

results = search(**args)
result_json = json.dumps(results, indent=2)

In [16]:
messages.extend(response.output)

messages.append({
    "type": "function_call_output",
    "call_id": call.call_id,
    "output": result_json,
})

In [17]:
messages

[{'role': 'user', 'content': 'How do I run Olama locally?'},
 ResponseFunctionToolCall(arguments='{"query":"Olama locally run install model serve local FAQ"}', call_id='call_9Nl5VZeLEQBEZO7u1YyYLz3M', name='search', type='function_call', id='fc_0d49fe1f4657922a006a29cd447d0881a2ac6672a1bf43f323', namespace=None, status='completed'),
 {'type': 'function_call_output',
  'call_id': 'call_9Nl5VZeLEQBEZO7u1YyYLz3M',
  'output': '[\n  {\n    "id": "cf93377279",\n    "course": "llm-zoomcamp",\n    "section": "Module 1: Introduction to LLMs and RAG",\n    "question": "ModuleNotFoundError on import docx in parse-faq.ipynb",\n    "answer": "The correct package name for `docx` is `python-docx`, not `docx`. Make sure to install the package using:\\n\\n```bash\\npip install python-docx\\n```"\n  },\n  {\n    "id": "d09e8d4843",\n    "course": "llm-zoomcamp",\n    "section": "Module 2: Agents",\n    "question": "Install MCP Inspector",\n    "answer": "1. Ensure Node.js is installed.\\n\\n2. To insta

In [18]:
response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)

response.output_text

'To run **Ollama locally**:\n\n1. **Install Ollama**\n   - macOS: download and install from https://ollama.com/download\n   - Windows: download the `.msi` installer from https://ollama.com/download\n   - Linux:\n     ```bash\n     curl -fsSL https://ollama.com/install.sh | sh\n     ```\n\n2. **Start a model locally**\n   ```bash\n   ollama run llama3\n   ```\n   This will download the model and open a local chat session.\n\n3. **Check that the local server is running**\n   ```bash\n   curl http://localhost:11434\n   ```\n   You should get a response from Ollama.\n\n4. **Use it from Python**\n   ```bash\n   pip install ollama\n   ```\n\n   Example:\n   ```python\n   import ollama\n\n   response = ollama.chat(\n       model=\'llama3\',\n       messages=[{"role": "user", "content": "Hello!"}]\n   )\n\n   print(response[\'message\'][\'content\'])\n   ```\n\nIf you want, I can also show you how to run a different Ollama model or connect it to your app.'

In [19]:
usage = response.usage
usage.input_tokens, usage.output_tokens

(1053, 251)

In [24]:
from utils import calculate_openai_price
result = calculate_openai_price(
    input_tokens=usage.input_tokens,
    output_tokens=usage.output_tokens,
    model="gpt-5.4-mini",
)

print("Total cost: $", round(result["total_cost"], 8))

Total cost: $ 0.00191925


## Agent loop

In [25]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches.

Try to expand your search by using new keywords
based on the results you get from the search.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

In [26]:
def make_call(call):
    args = json.loads(call.arguments)
    # Call the appropriate function based on the call name
    if call.name == "search":
        result = search(**args)
    # You can add more functions here as needed
    result_json = json.dumps(result, indent=2)

    return {
        "type": "function_call_output",
        "call_id": call.call_id,
        "output": result_json,
    }

In [27]:
question = "I just discovered the course. Can I join it?"

messages = [
    {"role": "developer", "content": instructions},
    {"role": "user", "content": question},
]

response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)

messages.extend(response.output)
has_function_calls = False

for item in response.output:
    if item.type == "function_call":
        print("function_call:", item.name, item.arguments)
        call_output = make_call(item)
        messages.append(call_output)
        has_function_calls = True

    elif item.type == "message":
        print("ASSISTANT:")
        print(item.content[0].text)

function_call: search {"query":"join course late enrollment discovered course can I join"}
function_call: search {"query":"course enrollment late join new student can I join"}
function_call: search {"query":"discovered the course can I join enrollment FAQ"}


In [28]:
it = 1

while True:
    print(f"iteration #{it}...")
    has_function_calls = False

    response = openai_client.responses.create(
        model="gpt-5.4-mini",
        input=messages,
        tools=[search_tool],
    )

    messages.extend(response.output)

    for item in response.output:
        if item.type == "function_call":
            print("function_call:", item.name, item.arguments)
            call_output = make_call(item)
            messages.append(call_output)
            has_function_calls = True

        elif item.type == "message":
            print("ASSISTANT:")
            print(item.content[0].text)

    it = it + 1
    if has_function_calls == False:
        break

iteration #1...
ASSISTANT:
Yes — you can still join the course.

If you want a certificate, though, you need to submit your project while submissions are still being accepted. Also, certificates are only available for the live cohort, not self-paced participation.

If you want, I can also explain how registration, homework submission, and certificates work.


In [32]:
from typing import Any, cast

def agent_loop(instructions: str, question: str, model: str = "gpt-5.4-mini") -> str:
    """
    Run an agent loop with tool calls until the model returns a final answer.

    Args:
        instructions: System/developer instructions for the assistant.
        question: User question.
        model: OpenAI model name.

    Returns:
        The last assistant text answer produced in the loop.
    """
    messages: list[Any] = [
        {"role": "developer", "content": instructions},
        {"role": "user", "content": question},
    ]

    it: int = 1
    last_answer: str = ""

    while True:
        print(f"iteration #{it}...")
        has_function_calls: bool = False

        response = openai_client.responses.create(
            model=model,
            input=cast(Any, messages),
            tools=cast(Any, [search_tool]),
        )

        messages.extend(response.output)

        for item in response.output:
            item_type = getattr(item, "type", None)

            if item_type == "function_call":
                print("function_call:", item.name, item.arguments)
                call_output: dict[str, str] = make_call(item)
                messages.append(call_output)
                has_function_calls = True

            elif item_type == "message":
                print("ASSISTANT:")
                for part in getattr(item, "content", []):
                    if getattr(part, "type", None) == "output_text":
                        last_answer = part.text
                        print(part.text)

        it += 1
        if not has_function_calls:
            break

    return last_answer


In [33]:
agent_loop(instructions, "How do I run Olama locally?")

iteration #1...
function_call: search {"query":"Olama local run install local Ollama run locally"}
function_call: search {"query":"Ollama local setup run model locally install"}
function_call: search {"query":"run Ollama locally course FAQ"}
iteration #2...
ASSISTANT:
To run Ollama locally:

1. **Install Ollama**
   - Go to: https://ollama.com/download
   - **macOS**: download and install the `.pkg`
   - **Windows**: download and install the `.msi`
   - **Linux**:
     ```bash
     curl -fsSL https://ollama.com/install.sh | sh
     ```

2. **Start a model locally**
   ```bash
   ollama run llama3
   ```
   This will download the model if needed and open a local chat interface.

3. **Check that the local server is running**
   ```bash
   curl http://localhost:11434
   ```
   You should get a response showing available models or server info.

4. **Use it from Python**
   ```bash
   pip install ollama
   ```

   Example:
   ```python
   import ollama

   response = ollama.chat(
       mod

'To run Ollama locally:\n\n1. **Install Ollama**\n   - Go to: https://ollama.com/download\n   - **macOS**: download and install the `.pkg`\n   - **Windows**: download and install the `.msi`\n   - **Linux**:\n     ```bash\n     curl -fsSL https://ollama.com/install.sh | sh\n     ```\n\n2. **Start a model locally**\n   ```bash\n   ollama run llama3\n   ```\n   This will download the model if needed and open a local chat interface.\n\n3. **Check that the local server is running**\n   ```bash\n   curl http://localhost:11434\n   ```\n   You should get a response showing available models or server info.\n\n4. **Use it from Python**\n   ```bash\n   pip install ollama\n   ```\n\n   Example:\n   ```python\n   import ollama\n\n   response = ollama.chat(\n       model=\'llama3\',\n       messages=[{"role": "user", "content": "Hello!"}]\n   )\n\n   print(response[\'message\'][\'content\'])\n   ```\n\nIf you want, I can also show you how to run Ollama in the background, use a different model, or co

In [34]:
agent_loop(instructions, "I just discovered the course. Can I still join it?")

iteration #1...
function_call: search {"query":"join course late enrollment discovered course can I still join"}
function_call: search {"query":"course enrollment late join FAQ discovered course"}
function_call: search {"query":"can I still join the course after it started FAQ"}
iteration #2...
ASSISTANT:
Yes — you can still join the course.

If you want a certificate, make sure you submit your project while submissions are still open. Also, certificates are only available if you finish with the live cohort, not in self-paced mode.

If you’d like, I can also help with:
- how to start if you’re joining late
- certificate requirements
- homework/project deadlines

Anything else you want to explore?


'Yes — you can still join the course.\n\nIf you want a certificate, make sure you submit your project while submissions are still open. Also, certificates are only available if you finish with the live cohort, not in self-paced mode.\n\nIf you’d like, I can also help with:\n- how to start if you’re joining late\n- certificate requirements\n- homework/project deadlines\n\nAnything else you want to explore?'

In [35]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results 
and then perform more searches. 

At the end, ask if there are other areas that the user wants to explore.
""".strip()

agent_loop(instructions, "I just discovered the course. Can I join it?")

iteration #1...
function_call: search {"query":"join the course after it has started enrollment late registration can I join discovered the course"}
iteration #2...
function_call: search {"query":"certificate project submission while accepting submissions live cohort peer review course discovered late FAQ"}
iteration #3...
ASSISTANT:
Yes — you can still join the course.

If you want a certificate, though, you need to submit your project while submissions are still open. Also, certificates are only available if you complete the course with the live cohort, not in self-paced mode.

If you’d like, I can also explain how registration, homework submission, and certificates work.


'Yes — you can still join the course.\n\nIf you want a certificate, though, you need to submit your project while submissions are still open. Also, certificates are only available if you complete the course with the live cohort, not in self-paced mode.\n\nIf you’d like, I can also explain how registration, homework submission, and certificates work.'

In [36]:
agent_loop(instructions, "what's queen gambit?")

iteration #1...
function_call: search {"query":"queen gambit chess opening queen's gambit what is it"}
iteration #2...
function_call: search {"query":"queen's gambit chess opening definition accepted declined common moves"}
iteration #3...
ASSISTANT:
The **Queen’s Gambit** is a classic **chess opening**.

It starts with:
1. **d4 d5**
2. **c4**

White offers the c-pawn to try to **pull Black’s d-pawn away** from the center, helping White gain control of the center squares.

A common reply is:
- **Queen’s Gambit Accepted (QGA):** Black takes the pawn with `dxc4`
- **Queen’s Gambit Declined (QGD):** Black does not take it and instead supports the center

So it’s called a “gambit” because White is **offering a pawn** to get a positional advantage, not necessarily expecting to keep it.

If you want, I can also explain the difference between **Queen’s Gambit Accepted** and **Declined** in simple terms.


'The **Queen’s Gambit** is a classic **chess opening**.\n\nIt starts with:\n1. **d4 d5**\n2. **c4**\n\nWhite offers the c-pawn to try to **pull Black’s d-pawn away** from the center, helping White gain control of the center squares.\n\nA common reply is:\n- **Queen’s Gambit Accepted (QGA):** Black takes the pawn with `dxc4`\n- **Queen’s Gambit Declined (QGD):** Black does not take it and instead supports the center\n\nSo it’s called a “gambit” because White is **offering a pawn** to get a positional advantage, not necessarily expecting to keep it.\n\nIf you want, I can also explain the difference between **Queen’s Gambit Accepted** and **Declined** in simple terms.'

In [37]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results 
and then perform more searches. 

The question has to be about the course or its logistics, offtopic questions 
shouldn't be answered. If the search returns nothing, it's likely an off-topic question.
If you can't answer the question using FAQ, don't do it yourself. Only use the 
facts from the FAQ database.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

agent_loop(instructions, "what's queen gambit?")

iteration #1...
function_call: search {"query":"queen gambit chess opening queen gambit"}
iteration #2...
function_call: search {"query":"queen gambit queen's gambit course faq"}
iteration #3...
ASSISTANT:
I couldn’t find a course/FAQ entry for “queen gambit,” so I can’t answer it from the course database.

If you meant something else related to the course, feel free to clarify, and I can look it up. Are there other areas you want to explore?


'I couldn’t find a course/FAQ entry for “queen gambit,” so I can’t answer it from the course database.\n\nIf you meant something else related to the course, feel free to clarify, and I can look it up. Are there other areas you want to explore?'